<a href="https://colab.research.google.com/github/bharath-hue/TASK_TNS_AIML/blob/main/Generative_AI_(Gen_AI).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np

# Generator
generator = tf.keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(100,)),
    layers.Dense(784, activation='sigmoid')
])

# Discriminator
discriminator = tf.keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(784,)),
    layers.Dense(1, activation='sigmoid')
])

# Compile discriminator
discriminator.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# GAN Model
discriminator.trainable = False

gan = tf.keras.Sequential([
    generator,
    discriminator
])

gan.compile(
    optimizer='adam',
    loss='binary_crossentropy'
)

# Dummy Training Loop
for epoch in range(100):
    noise = np.random.normal(0, 1, (32, 100))

    fake_images = generator.predict(noise, verbose=0)

    real_images = np.random.rand(32, 784)

    X = np.vstack([real_images, fake_images])
    y = np.vstack([np.ones((32,1)), np.zeros((32,1))])

    discriminator.trainable = True
    discriminator.train_on_batch(X, y)

    discriminator.trainable = False

    noise = np.random.normal(0,1,(32,100))
    gan.train_on_batch(noise, np.ones((32,1)))

print("GAN Training Complete")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


GAN Training Complete


In [2]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model

input_dim = 784
latent_dim = 2

# Encoder
inputs = Input(shape=(input_dim,))
h = Dense(128, activation='relu')(inputs)

z_mean = Dense(latent_dim)(h)
z_log_var = Dense(latent_dim)(h)

# Sampling
def sampling(args):
    z_mean, z_log_var = args
    epsilon = tf.random.normal(shape=tf.shape(z_mean))
    return z_mean + tf.exp(0.5 * z_log_var) * epsilon

z = tf.keras.layers.Lambda(sampling)(
    [z_mean, z_log_var]
)

# Decoder
decoder_h = Dense(128, activation='relu')
decoder_out = Dense(input_dim,
                    activation='sigmoid')

h_decoded = decoder_h(z)
outputs = decoder_out(h_decoded)

vae = Model(inputs, outputs)

vae.compile(
    optimizer='adam',
    loss='binary_crossentropy'
)

vae.summary()

# Dummy data
X = tf.random.uniform((1000,784))

vae.fit(
    X,
    X,
    epochs=3,
    batch_size=32
)

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 784)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 128)       │    100,480 │ input_layer_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 2)         │        258 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 2)         │        258 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 2)         │          0 │ dense_5[0][0],    │
│                     │                   │            │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 128)       │        384 │ lambda[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 784)       │    101,136 │ dense_7[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 202,516 (791.08 KB)

 Trainable params: 202,516 (791.08 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/3
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.6932
Epoch 2/3
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.6930
Epoch 3/3
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.6927


In [3]:
import tensorflow as tf
from tensorflow.keras.layers import (
    Embedding,
    MultiHeadAttention,
    LayerNormalization,
    Dense,
    Input
)
from tensorflow.keras.models import Model

vocab_size = 10000
max_len = 50
embed_dim = 64
num_heads = 4

# Input
inputs = Input(shape=(max_len,))

# Token Embedding
x = Embedding(
    vocab_size,
    embed_dim
)(inputs)

# Self-Attention
attention = MultiHeadAttention(
    num_heads=num_heads,
    key_dim=embed_dim
)

attn_output = attention(x, x)

x = LayerNormalization()(x + attn_output)

# Feed Forward Network
ffn = Dense(
    128,
    activation='relu'
)(x)

ffn = Dense(embed_dim)(ffn)

x = LayerNormalization()(x + ffn)

# Vocabulary Prediction
outputs = Dense(
    vocab_size,
    activation='softmax'
)(x)

model = Model(inputs, outputs)

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy'
)

model.summary()

# Dummy Training Data
X = tf.random.uniform(
    (100, max_len),
    maxval=vocab_size,
    dtype=tf.int32
)

y = tf.random.uniform(
    (100, max_len),
    maxval=vocab_size,
    dtype=tf.int32
)

model.fit(
    X,
    y,
    epochs=2,
    batch_size=8
)

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 50, 64)    │    640,000 │ input_layer_4[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 50, 64)    │     66,368 │ embedding[0][0],  │
│ (MultiHeadAttentio… │                   │            │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 50, 64)    │          0 │ embedding[0][0],  │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 50, 64)    │        128 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 50, 128)   │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 50, 64)    │      8,256 │ dense_9[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 50, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ dense_10[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 50, 64)    │        128 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 50, 10000) │    650,000 │ layer_normalizat… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,373,200 (5.24 MB)

 Trainable params: 1,373,200 (5.24 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/2
13/13 ━━━━━━━━━━━━━━━━━━━━ 7s 240ms/step - loss: 9.2186
Epoch 2/2
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 125ms/step - loss: 8.7431
